# 01 · Coleta e camada Bronze

**Objetivo da etapa:** obter o conjunto de dados de origem, validar a integridade, registrar a
procedência (data, volume, formato, licença) e persistir uma cópia fiel na nuvem, na camada **Bronze**.

| Item | Valor |
|---|---|
| Conjunto de dados | *Brazilian E-Commerce Public Dataset by Olist* |
| Fonte oficial | Kaggle — `olistbr/brazilian-ecommerce` |
| Licença | CC BY-NC-SA 4.0: uso não comercial, com atribuição e compartilhamento pela mesma licença |
| Formato original | 9 arquivos CSV (UTF-8, separador vírgula, com cabeçalho) |
| Período coberto | set/2016 a out/2018 |
| Dados pessoais | Anonimizados na origem: clientes e vendedores aparecem como hashes, e a localização só até o prefixo de 5 dígitos do CEP |

**Estratégia de coleta (reprodutível):**
1. tenta baixar da fonte oficial com a biblioteca `kagglehub`;
2. se não houver acesso ao Kaggle, por exemplo sem credenciais, usa um espelho público com os
   mesmos 9 arquivos (Hugging Face);
3. **em ambos os casos**, cada arquivo é validado contra a contagem de linhas e as colunas
   oficiais. Se algo divergir, a coleta é interrompida.

Por respeito à licença e por boa prática, os dados brutos **não** são versionados no GitHub.
O que vai para o repositório é este notebook, que reproduz a coleta.

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    # O Spark precisa de Java: instala um JDK apenas se a máquina não tiver um
    if subprocess.run(["bash", "-c", "command -v java"], capture_output=True).returncode != 0:
        subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

In [ ]:
import hashlib, shutil, urllib.request
from datetime import datetime, timezone
from pyspark.sql import functions as F

C.banner("01 · Coleta e camada Bronze")

# Contrato de dados: arquivo -> (tabela bronze, nº de linhas oficial, colunas esperadas)
CONTRATO = {
    "olist_orders_dataset.csv": ("orders", 99_441, ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]),
    "olist_customers_dataset.csv": ("customers", 99_441, ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]),
    "olist_order_items_dataset.csv": ("order_items", 112_650, ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"]),
    "olist_order_payments_dataset.csv": ("order_payments", 103_886, ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]),
    "olist_order_reviews_dataset.csv": ("order_reviews", 99_224, ["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message", "review_creation_date", "review_answer_timestamp"]),
    "olist_products_dataset.csv": ("products", 32_951, ["product_id", "product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]),
    "olist_sellers_dataset.csv": ("sellers", 3_095, ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]),
    "olist_geolocation_dataset.csv": ("geolocation", 1_000_163, ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"]),
    "product_category_name_translation.csv": ("category_translation", 71, ["product_category_name", "product_category_name_english"]),
}
KAGGLE_ID = "olistbr/brazilian-ecommerce"
MIRROR = "https://huggingface.co/datasets/aviahYadler/Olist_Ecommerce_Dataset/resolve/main"

## 1. Download

In [ ]:
dest = f"{C.RAW_DIR}/olist"
os.makedirs(dest, exist_ok=True)
fonte = None

try:  # 1ª opção: fonte oficial (Kaggle)
    import kagglehub
    pasta = kagglehub.dataset_download(KAGGLE_ID)
    for arq in CONTRATO:
        shutil.copy(os.path.join(pasta, arq), f"{dest}/{arq}")
    fonte = f"Kaggle ({KAGGLE_ID}) via kagglehub"
except Exception as e:  # 2ª opção: espelho público
    print(f"Kaggle indisponível ({type(e).__name__}); usando espelho público.")
    for arq in CONTRATO:
        urllib.request.urlretrieve(f"{MIRROR}/{arq}", f"{dest}/{arq}")
    fonte = f"Espelho público do Kaggle {KAGGLE_ID}: {MIRROR}"

data_coleta = datetime.now(timezone.utc)
print("Fonte utilizada:", fonte)
print("Data da coleta (UTC):", data_coleta.isoformat(timespec="seconds"))

## 2. Validação contra o contrato de dados
A leitura usa `multiLine=True`, porque os comentários das avaliações contêm quebras de linha
dentro de campos entre aspas. Todas as colunas são lidas como **texto**: a tipagem acontece
na camada Silver, para que nenhuma informação se perca na Bronze, como os zeros à esquerda do CEP.

In [ ]:
def ler_csv(caminho):
    return (spark.read.option("header", True).option("multiLine", True)
            .option("quote", '"').option("escape", '"').option("inferSchema", False)
            .csv(caminho))

def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

manifesto, erros = [], []
for arq, (tabela, n_oficial, colunas) in CONTRATO.items():
    df = ler_csv(f"{dest}/{arq}")
    n = df.count()
    ok_linhas, ok_colunas = n == n_oficial, df.columns == colunas
    if not (ok_linhas and ok_colunas):
        erros.append(arq)
    manifesto.append({
        "arquivo": arq, "tabela_bronze": tabela, "formato": "CSV",
        "bytes": os.path.getsize(f"{dest}/{arq}"), "sha256": sha256(f"{dest}/{arq}"),
        "linhas": n, "linhas_oficiais": n_oficial, "colunas": len(df.columns),
        "valido": ok_linhas and ok_colunas})

import pandas as pd
man = pd.DataFrame(manifesto)
display(man[["arquivo", "linhas", "linhas_oficiais", "colunas", "bytes", "valido"]])
assert not erros, f"Arquivos fora do contrato: {erros}"
print(f"Volume total coletado: {man.bytes.sum()/1e6:,.1f} MB | {man.linhas.sum():,} linhas | 9/9 arquivos válidos")

C.save_json({"fonte": fonte, "dataset": KAGGLE_ID, "licenca": "CC BY-NC-SA 4.0",
             "data_coleta_utc": data_coleta.isoformat(timespec="seconds"),
             "run_id": C.RUN_ID, "arquivos": manifesto}, f"{C.RAW_DIR}/manifesto_coleta.json")

## 3. Carga na camada Bronze (Delta Lake)
A Bronze é uma cópia fiel da origem, com todas as colunas como texto. Cada linha recebe três
**colunas de linhagem**:
- `_arquivo_origem`: arquivo de onde a linha veio;
- `_data_coleta`: momento da coleta;
- `_run_id`: identificador da execução do pipeline.

In [ ]:
for arq, (tabela, _, _) in CONTRATO.items():
    with C.Stage("01_coleta", f"bronze.{tabela}") as st:
        df = (ler_csv(f"{dest}/{arq}")
              .withColumn("_arquivo_origem", F.lit(arq))
              .withColumn("_data_coleta", F.lit(data_coleta.replace(tzinfo=None)).cast("timestamp"))
              .withColumn("_run_id", F.lit(C.RUN_ID)))
        st.rows = C.write_table(df, "bronze", tabela, comment=f"coleta {fonte}")

## 4. Verificação da persistência
A verificação lê de volta, **a partir do Google Drive**, cada tabela gravada e confere a contagem
de linhas.

In [ ]:
for arq, (tabela, n_oficial, _) in CONTRATO.items():
    n = spark.read.format("delta").load(f"{C.CLOUD_BASE}/bronze/{tabela}").count()
    print(f"bronze.{tabela:<22} {n:>10,} linhas  {'OK' if n == n_oficial else 'DIVERGENTE'}")